In [1]:
import numpy as np
import sys
sys.path.append('/home/tschiwy/zpenoyre')
import gaiascanlaw as gs
from astropy.table import Table

In [2]:
pth="/home/tschiwy/gaiaprep/signature/carmenes/M_dwarfs.txt"
data=Table.read(pth,format="ascii")
stars=data["hostname"]
planets=data["pl_name"]
gaia_ids=data["gaia_dr3_id"]

In [3]:
gaia_ids=[str(d[9:]) for d in data["gaia_dr3_id"]]

In [4]:
from astropy.time import Time

GOST=Table(gs.scanlaw(stand[0],stand[1],tend=gs.tdr4,obstype="astrometry",parallax_factor=True)[:-1],names=["t","scan","plx_al"])

In [5]:
from astroquery.gaia import Gaia
tbl = Table(
    names=["ra", "dec", "parallax", "pmra", "pmdec","rv"],
    dtype=[float, float,float, float, float,float]
)

g= Table(
    names=["t", "scan", "plx_al", "plx_ac"],
    dtype=[list, list,list,list]
)
for n,gid in zip(stars,gaia_ids):

    query = f"""
    SELECT *
    FROM gaiadr3.gaia_source
    WHERE source_id = {gid}
    """

    job = Gaia.launch_job(query)
    result = job.get_results()
    stand=[np.radians(result["ra"][0]), np.radians(result["dec"][0]), result["parallax"][0], result["pmra"][0], result["pmdec"][0], result["radial_velocity"][0]]
    stand_err=[result["ra_error"][0], result["dec_error"][0], result["parallax_error"][0], result["pmra_error"][0], result["pmdec_error"][0], result["radial_velocity_error"][0]]

    tbl.add_row(stand)

    GOST=Table(gs.scanlaw(stand[0],stand[1],tend=gs.tdr4,obstype="astrometry",parallax_factor=True)[:-1],names=["t","scan","plx_al"])
    GOST["t"]=Time(GOST["t"],format="jyear").jd

    filename="GOST/"+n+"_GOST.txt"

    GOST.write(filename,format="ascii",overwrite=True)

    # Read existing lines
    with open(filename, "r", newline="") as f:
        rows = f.readlines()
    
    #print(n)
    #print(stand)

    stand_str = ", ".join(str(float(x)) for x in stand)
    stand_err_str = ", ".join(str(float(x)) for x in stand_err)
    # Write header + old lines back
    with open(filename, "w", newline="") as f:

        f.write("# DR3 solution:" +stand_str+"\n# Error:" + stand_err_str + "\n")
        f.writelines(rows)


The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.


import pyvo

service = pyvo.dal.TAPService(
    "https://exoplanetarchive.ipac.caltech.edu/TAP"
)

# gaia_ids: bare Gaia source_id ints/strings, e.g. [5925209583053212800, ...]
id_clauses = " OR ".join(
    f"gaia_dr3_id LIKE '%{gid}'"
    for gid in gaia_ids
)

query = f"""
SELECT
    pl_name,
    hostname,
    gaia_dr3_id,
    sy_plx,
    st_mass,
    st_rotp,
    st_rotperr1,
    st_rotperr2,
    st_rotp_reflink,
    pl_bmassj,
    pl_orbper,
    pl_orbeccen,
    pl_orblper,
    pl_orbincl,
    pl_orbinclerr1,
    pl_orbinclerr2,
    pl_orbtper,
    pl_tranmid,
    pl_orbsmax
FROM ps
WHERE {id_clauses}
"""

result = service.search(query)
table0 = result.to_table()
table0.write("exoplanetarchive.txt", format="ascii", overwrite=True)

planet=table0["hostname",'pl_name',"sy_plx",
 'pl_orbper',
 'pl_orbeccen',
 'pl_orblper',
 'pl_orbincl',
 'pl_orbtper',
 'pl_orbsmax',
 "st_mass",
 "pl_bmassj"]
planet=planet[np.isin(planet["pl_name"],pl)]
planet.write("orbital_pars.txt",format="ascii",overwrite=True)

def astar_mas(a_au, mp_mjup, Mstar_msun, plx_mas):
    MJUP_MSUN = 9.5479e-4
    mp = np.asarray(mp_mjup) * MJUP_MSUN          # -> Msun
    a_star_au = a_au * mp / (Mstar_msun + mp)
    return a_star_au * plx_mas

mtp=[]
for obj in planet:
    #print(np.where(planet["hostname"]==obj["hostname"])[0])
    mtp.append(np.where(planet["hostname"]==obj["hostname"])[0])